In [1]:
!pip install pandas langchain-core langchain-milvus langchain_openai "pymilvus>=2.5.0,<2.6.0"

In [2]:
import pandas as pd
import zipfile
import urllib.request
from langchain_core.documents import Document
from langchain_milvus import Milvus
import requests
from typing import List
from langchain_core.embeddings import Embeddings
from pymilvus import MilvusClient
import os
import json

In [3]:
url_cwe = "https://cwe.mitre.org/data/csv/699.csv.zip"
zip_path = "699.csv.zip"
extract_cwe_dir = "cwe_data"
csv_path = extract_cwe_dir + "/699.csv"
osv_pypi_url = "https://osv-vulnerabilities.storage.googleapis.com/PyPI/all.zip"
zip_pypi_path = "osv_pypi.zip"
extract_osv_dir = "osv_data"

print(f"Descargando base de datos CWE de MITRE desde {url_cwe}...")
urllib.request.urlretrieve(url_cwe, zip_path)

print(f"Descargando base de datos de PyPi OSV desde {osv_pypi_url}...")
response = requests.get(osv_pypi_url)
with open(zip_pypi_path, 'wb') as f:
    f.write(response.content)

osv_maven_url = "https://osv-vulnerabilities.storage.googleapis.com/Maven/all.zip"
zip_maven_path = "osv_maven.zip"

print(f"Descargando base de datos de Maven OSV desde {osv_maven_url}...")
response = requests.get(osv_maven_url)
with open(zip_maven_path, 'wb') as f:
    f.write(response.content)

Descargando base de datos CWE de MITRE desde https://cwe.mitre.org/data/csv/699.csv.zip...
Descargando base de datos de PyPi OSV desde https://osv-vulnerabilities.storage.googleapis.com/PyPI/all.zip...
Descargando base de datos de Maven OSV desde https://osv-vulnerabilities.storage.googleapis.com/Maven/all.zip...


In [4]:
print("Extrayendo archivos CWE...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_cwe_dir)

df = pd.read_csv(csv_path)

print("Extrayendo archivos PyPi JSON...")
with zipfile.ZipFile(zip_pypi_path, 'r') as zip_ref:
    zip_ref.extractall(extract_osv_dir)

print("Extrayendo archivos Maven JSON...")
with zipfile.ZipFile(zip_maven_path, 'r') as zip_ref:
    zip_ref.extractall(extract_osv_dir)

archivos_json = [f for f in os.listdir(extract_osv_dir) if f.endswith('.json')]

processed_cwe_docs = []
processed_osv_docs = []

Extrayendo archivos CWE...
Extrayendo archivos PyPi JSON...
Extrayendo archivos Maven JSON...


In [5]:
print("Procesando vulnerabilidades y mitigaciones...")
for index, row in df.iterrows():
    cwe_id = str(row.get('CWE - ID', ''))
    name = str(row.get('Name', ''))
    description = str(row.get('Description', ''))
    extended_desc = str(row.get('Extended Description', ''))

    mitigations = str(row.get('Potential Mitigations', 'No hay mitigación estándar registrada.'))

    if description == 'nan': description = ""
    if extended_desc == 'nan': extended_desc = ""
    if mitigations == 'nan': mitigations = "Revisar la lógica de negocio y sanitizar entradas."

    contenido_semantico = f"""
    Identificador: CWE-{cwe_id}
    Nombre del problema: {name}

    Descripción de la falla en el código:
    {description}
    {extended_desc}

    Estrategias de Mitigación y Remediación (Cómo solucionarlo):
    {mitigations}
    """

    doc = Document(
        page_content=contenido_semantico,
        metadata={
            "source": "mitre_cwe",
            "cwe_id": f"CWE-{cwe_id}",
            "name": name
        }
    )
    processed_cwe_docs.append(doc)

print(f"¡Listo! Se procesaron {len(processed_cwe_docs)} estrategias de mitigación CWE.")

Procesando vulnerabilidades y mitigaciones...
¡Listo! Se procesaron 399 estrategias de mitigación CWE.


In [6]:
print(f"Procesando {len(archivos_json)} vulnerabilidades. Esto puede tomar un momento...")

for filename in archivos_json:
#for filename in archivos_json[:1000]:
    filepath = os.path.join(extract_osv_dir, filename)

    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

        vuln_id = data.get("id", "Desconocido")
        description = data.get("details", data.get("summary", "Sin descripción detallada."))

        package_name = "Desconocido"
        fixed_version = "No hay parche oficial disponible. Considerar mitigaciones alternativas."

        if "affected" in data and len(data["affected"]) > 0:
            affected_node = data["affected"][0]
            if "package" in affected_node:
                package_name = affected_node["package"].get("name", "Desconocido")

            if "ranges" in affected_node:
                for r in affected_node["ranges"]:
                    if "events" in r:
                        for event in r["events"]:
                            if "fixed" in event:
                                fixed_version = event["fixed"]
                                break

        contenido_semantico = f"""
        Identificador de Vulnerabilidad (CVE/GHSA): {vuln_id}
        Paquete Afectado: {package_name}

        Descripción técnica del fallo:
        {description}

        Solución y Remediación:
        Para solucionar esta vulnerabilidad, debes actualizar el paquete '{package_name}' a la versión {fixed_version} o superior.
        """

        if len(description) > 50:
            doc = Document(
                page_content=contenido_semantico,
                metadata={
                    "source": "osv_dev",
                    "vuln_id": vuln_id,
                    "package": package_name
                }
            )
            processed_osv_docs.append(doc)

print(f"¡Listo! Se procesaron {len(processed_osv_docs)} vulnerabilidades accionables.")

Procesando 27326 vulnerabilidades. Esto puede tomar un momento...
¡Listo! Se procesaron 27298 vulnerabilidades accionables.


In [7]:
milvus_uri = "http://milvus-service:19530"
coleccion = "company_coding_standards"

client = MilvusClient(uri=milvus_uri)

if client.has_collection(collection_name=coleccion):
    client.drop_collection(collection_name=coleccion)
    print(f"Colección '{coleccion}' eliminada correctamente.")

Colección 'company_coding_standards' eliminada correctamente.


In [8]:
class LlamaStackGraniteEmbeddings(Embeddings):
    """
    Adaptador personalizado para enviar textos puros a Llama Stack,
    evitando los bugs de arrays del SDK de OpenAI.
    """

    def __init__(self, base_url: str, model: str):
        self.base_url = base_url
        self.model = model

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        vectores = []
        # Enviamos los textos uno por uno como strings puros
        for text in texts:
            # Aseguramos que sea un string válido
            texto_limpio = str(text).strip()
            payload = {
                "model": self.model,
                "input": texto_limpio
            }
            response = requests.post(
                f"{self.base_url}/embeddings",  # Endpoint /v1/embeddings
                json=payload,
                headers={"Content-Type": "application/json"}
            )
            # Si hay un error HTTP, nos lo mostrará claramente
            response.raise_for_status() 
            # Extraemos el vector de la respuesta de Llama Stack
            vector = response.json()["data"][0]["embedding"]
            vectores.append(vector)
        return vectores

    def embed_query(self, text: str) -> List[float]:
        # Para LangChain, un query es solo un documento de 1 elemento
        return self.embed_documents([text])[0]


embeddings = LlamaStackGraniteEmbeddings(
    model="sentence-transformers/ibm-granite/granite-embedding-125m-english", 
    base_url="http://llama-stack-milvus-remote-service.triage-analysis-sast.svc.cluster.local:8321/v1"
)

In [9]:
print("Conectando a Milvus con el nuevo MilvusClient...")

vector_store = Milvus(
    embedding_function=embeddings,
    connection_args={"uri": milvus_uri},
    collection_name=coleccion,
    index_params={"index_type": "FLAT", "metric_type": "L2"},
    auto_id=True,
    enable_dynamic_field=True,
)

print("Ingestando vectores de cwe documents en Milvus mediante LangChain...")

vectorstore_cwe = Milvus.from_documents(
    documents=processed_cwe_docs,
    embedding=embeddings,
    connection_args={"uri": milvus_uri},
    collection_name=coleccion,
    drop_old=True,
    auto_id=True,
    enable_dynamic_field=True,
)
print("✅ ¡Ingesta de cwe documents completada con éxito!")

Conectando a Milvus con el nuevo MilvusClient...
Ingestando vectores de cwe documents en Milvus mediante LangChain...
✅ ¡Ingesta de cwe documents completada con éxito!


In [10]:
print("Ingestando vectores de vulnerabilidades de librerias en Milvus mediante LangChain...")

vectorstore_osv = Milvus.from_documents(
    documents=processed_osv_docs,
    embedding=embeddings,
    connection_args={"uri": milvus_uri},
    collection_name=coleccion,
    drop_old=True,
    auto_id=True,
    enable_dynamic_field=True,
)

print("✅ ¡Ingesta de vulnerabilidades de librerías completada con éxito!")

Ingestando vectores de vulnerabilidades de librerias en Milvus mediante LangChain...
✅ ¡Ingesta de vulnerabilidades de librerías completada con éxito!


In [13]:
question = "CVE-2021-44228"

search_res = client.search(
    collection_name=coleccion,
    data=[
        embeddings.embed_query(question)
    ],  # Use the `emb_text` function to convert the question to an embedding vector
    limit=3,  # Return top 3 results
    search_params={"metric_type": "L2", "params": {}},  # Inner product distance
    output_fields=["text"],  # Return the text field
)

import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

[
    [
        "\n        Identificador de Vulnerabilidad (CVE/GHSA): MAL-2024-4886\n        Paquete Afectado: cffii\n\n        Descripci\u00f3n t\u00e9cnica del fallo:\n        \n---\n_-= Per source details. Do not edit below this line.=-_\n\n\n        Soluci\u00f3n y Remediaci\u00f3n:\n        Para solucionar esta vulnerabilidad, debes actualizar el paquete 'cffii' a la versi\u00f3n No hay parche oficial disponible. Considerar mitigaciones alternativas. o superior.\n        ",
        0.22337910532951355
    ],
    [
        "\n        Identificador de Vulnerabilidad (CVE/GHSA): MAL-2025-48900\n        Paquete Afectado: test422\n\n        Descripci\u00f3n t\u00e9cnica del fallo:\n        \n---\n_-= Per source details. Do not edit below this line.=-_\n\n\n        Soluci\u00f3n y Remediaci\u00f3n:\n        Para solucionar esta vulnerabilidad, debes actualizar el paquete 'test422' a la versi\u00f3n No hay parche oficial disponible. Considerar mitigaciones alternativas. o superior.\n   